# Demo Alur Lengkap Pemrosesan Data NLP

In [1]:
import sys
import os

# Fix encoding di Windows (hanya untuk terminal biasa, bukan Jupyter)
if sys.platform == 'win32':
    import io
    if hasattr(sys.stdout, 'buffer'):
        sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8')

# Tambahkan root direktori ke path
# Di Jupyter notebook, __file__ tidak terdefinisi, jadi gunakan os.getcwd()
if '__file__' in globals():
    # Jika file ini dijalankan dari notebooks/ folder, root_dir adalah parentnya
    file_dir = os.path.dirname(os.path.abspath(__file__))
    if os.path.basename(file_dir) == 'notebooks':
        root_dir = os.path.dirname(file_dir)
    else:
        root_dir = file_dir
else:
    cwd = os.getcwd()
    if os.path.basename(cwd) == 'notebooks':
        root_dir = os.path.dirname(cwd)
    else:
        root_dir = cwd
sys.path.insert(0, root_dir)

from nlp.processor import proses_nlp, split_multi_entries, _apply_multi_overrides
from nlp.normalizer import koreksi_teks
from nlp.extractor import ekstrak_entitas
from nlp.dictionaries import KAMUS_ALIAS, NORMALIZATION_DICT, DAFTAR_KATA_KUNCI
from core.master_data import cari_harga_default, format_rupiah, parse_rupiah
import re

print("[OK] Semua modul berhasil diimport!")
print("[OK] Sistem siap digunakan!")

[OK] Semua modul berhasil diimport!
[OK] Sistem siap digunakan!


## Langkah 1: Input Data

In [2]:
# Input contoh dengan 3 entri, nama Pak Andi, menggunakan "lolipop"
INPUT_TEKS = "pak anDi Pesan l0lipp 10 dus, mses 5 bks, will0 3 dus belum lunwas semua"

print(f"\nInput: {INPUT_TEKS}")


Input: pak anDi Pesan l0lipp 10 dus, mses 5 bks, will0 3 dus belum lunwas semua


## Langkah 2: Pemisahan Multi-Entry

In [3]:
entries = split_multi_entries(INPUT_TEKS)

print(f"\nJumlah entri: {len(entries)}")
for i, entry in enumerate(entries, 1):
    print(f"\nEntri {i}:")
    print(f"  Teks: {entry}")


Jumlah entri: 3

Entri 1:
  Teks: pak anDi Pesan l0lipp 10 dus

Entri 2:
  Teks: mses 5 bks

Entri 3:
  Teks: will0 3 dus belum lunwas semua


## Langkah 3: Text Cleaning

In [4]:
entri_contoh = entries[0]

print(f"\n1. Teks asli: {entri_contoh}")

teks_clean = re.sub(r'[^\w\s]', '', entri_contoh)
print(f"2. Setelah cleaning: {teks_clean}")


1. Teks asli: pak anDi Pesan l0lipp 10 dus
2. Setelah cleaning: pak anDi Pesan l0lipp 10 dus


## Langkah 4: Lowercase

In [5]:
print(f"\n1. Teks sebelumnya: {teks_clean}")

teks_bersih = teks_clean.lower()
print(f"2. Setelah lowercase: {teks_bersih}")


1. Teks sebelumnya: pak anDi Pesan l0lipp 10 dus
2. Setelah lowercase: pak andi pesan l0lipp 10 dus


## Langkah 5: Tokenization

In [6]:
tokens = teks_bersih.split()

print(f"\nTeks: {teks_bersih}")
print(f"\nTokens:")
for i, token in enumerate(tokens, 1):
    print(f"  Token {i}: '{token}'")


Teks: pak andi pesan l0lipp 10 dus

Tokens:
  Token 1: 'pak'
  Token 2: 'andi'
  Token 3: 'pesan'
  Token 4: 'l0lipp'
  Token 5: '10'
  Token 6: 'dus'


## Langkah 6: Normalization & Fuzzy Matching

In [7]:
print("\n--- Bagian A: DICTIONARY LOOKUP & PREPROCESSING ---")
print("\nContoh KAMUS_ALIAS (nama barang):")
contoh_barang = list(KAMUS_ALIAS.items())[:10]
for typo, benar in contoh_barang:
    print(f"  '{typo}' menjadi '{benar}'")

print("\n--- Bagian B: LANGKAH-LANGKAH KOREKSI TYPO SEPERTI SISTEM NYATA ---")
from rapidfuzz import fuzz, process
from nlp.dictionaries import DAFTAR_KATA_KUNCI
import re

# Proses satu per satu entri untuk menunjukkan alur
for idx_entri, entry in enumerate(entries, 1):
    print(f"\n=== MENGKOREKSI ENTRI {idx_entri}: '{entry}' ===")
    teks_proses = entry
    
    # PASS 0 & 1: Preprocess angka 0→o, 1→l
    print(f"\n1. Preprocessing angka 0→o dan 1→l:")
    print(f"   Sebelum: '{teks_proses}'")
    teks_proses = re.sub(r'(\b[a-zA-Z]+)([01])\b', r'\1', teks_proses, flags=re.IGNORECASE)
    teks_proses = re.sub(r'([a-zA-Z])0([a-zA-Z]*)', r'\1o\2', teks_proses, flags=re.IGNORECASE)
    teks_proses = re.sub(r'([a-zA-Z])1([a-zA-Z]*)', r'\1l\2', teks_proses, flags=re.IGNORECASE)
    words_pre = teks_proses.split()
    normalized_words = []
    for w in words_pre:
        w_low = w.lower()
        if not w.isdigit():
            w_normalized = w_low.replace('0', 'o').replace('1', 'l')
            normalized_words.append(w_normalized)
        else:
            normalized_words.append(w)
    teks_proses = " ".join(normalized_words)
    print(f"   Sesudah: '{teks_proses}'")
    
    # PASS 1.5: KAMUS_ALIAS lookup
    print(f"\n2. Cek KAMUS_ALIAS:")
    words_pre = teks_proses.split()
    replaced_words = []
    ignore_words = {"pak", "bu", "mas", "atas", "nama", "hari", "bulan", "tahun", "kemarin", "besok", "lusa", "minggu", "belum", "sudah", "semua", "setengah", "cicil", "transfer", "tunai", "cash", "dp", "kontan", "langsung", "lunas", "hutang", "utang", "bayar", "dibayar", "ngutang", "kasbon"}
    for w in words_pre:
        w_low = w.lower()
        if w_low in ignore_words:
            replaced_words.append(w)
        elif w_low in KAMUS_ALIAS:
            print(f"   ✅ '{w_low}' ditemukan di KAMUS_ALIAS → '{KAMUS_ALIAS[w_low]}'")
            replaced_words.append(KAMUS_ALIAS[w_low])
        else:
            replaced_words.append(w)
    teks_proses = " ".join(replaced_words)
    print(f"   Hasil: '{teks_proses}'")
    
    # PASS 3: Fuzzy matching untuk sisa kata
    print(f"\n3. Fuzzy Matching untuk kata yang belum ditemukan:")
    words = teks_proses.split()
    corrected_words = []
    i = 0
    while i < len(words):
        w_lower = words[i].lower()
        match_found = False
        # Cek status keywords typo (misal: lunwas → lunas)
        status_keywords = {"lunas", "hutang", "belum", "sudah", "setengah"}
        if w_lower not in ignore_words and len(w_lower) >=4 and not w_lower.isdigit():
            status_res = process.extractOne(w_lower, list(status_keywords), scorer=fuzz.ratio)
            if status_res and status_res[1] >=80:
                print(f"   ✅ '{words[i]}' mirip dengan '{status_res[0]}' (similarity {status_res[1]:.1f}%)")
                corrected_words.append(status_res[0])
                i +=1
                continue
        # Cek unigram fuzzy matching
        if len(words[i]) >=4 and not words[i].isdigit() and w_lower not in ignore_words:
            res_1 = process.extractOne(words[i], DAFTAR_KATA_KUNCI, scorer=fuzz.ratio)
            if res_1 and res_1[1] >=85:
                terbaik_1 = res_1[0]
                skor_1 = res_1[1]
                if terbaik_1 in KAMUS_ALIAS:
                    print(f"   ✅ '{words[i]}' mirip dengan '{terbaik_1}' → '{KAMUS_ALIAS[terbaik_1]}' (similarity {skor_1:.1f}%)")
                    corrected_words.append(KAMUS_ALIAS[terbaik_1])
                else:
                    print(f"   ✅ '{words[i]}' mirip dengan '{terbaik_1}' (similarity {skor_1:.1f}%)")
                    corrected_words.append(terbaik_1)
                match_found = True
                i +=1
                continue
        if not match_found:
            corrected_words.append(words[i])
            i +=1
    teks_final = " ".join(corrected_words)
    print(f"\n4. Hasil akhir koreksi: '{teks_final}'")

print("\n--- Bagian C: RINGKASAN HASIL KOREKSI SEMUA ENTRI ---")
for i, entry in enumerate(entries, 1):
    teks_dikoreksi = koreksi_teks(entry)
    print(f"\nEntri {i}:")
    print(f"  Sebelum: '{entry}'")
    print(f"  Sesudah: '{teks_dikoreksi}'")


--- Bagian A: DICTIONARY LOOKUP & PREPROCESSING ---

Contoh KAMUS_ALIAS (nama barang):
  'permen coklat' menjadi 'Permen Coklat'
  'coklat' menjadi 'Permen Coklat'
  'millo' menjadi 'Permen Coklat'
  'permen millo' menjadi 'Permen Coklat'
  'prmen coklat' menjadi 'Permen Coklat'
  'permen cokelat' menjadi 'Permen Coklat'
  'prmen' menjadi 'Permen Coklat'
  'willo' menjadi 'Willo'
  'permen willo' menjadi 'Willo'
  'wilo' menjadi 'Willo'

--- Bagian B: LANGKAH-LANGKAH KOREKSI TYPO SEPERTI SISTEM NYATA ---

=== MENGKOREKSI ENTRI 1: 'pak anDi Pesan l0lipp 10 dus' ===

1. Preprocessing angka 0→o dan 1→l:
   Sebelum: 'pak anDi Pesan l0lipp 10 dus'
   Sesudah: 'pak andi pesan lolipp 10 dus'

2. Cek KAMUS_ALIAS:
   Hasil: 'pak andi pesan lolipp 10 dus'

3. Fuzzy Matching untuk kata yang belum ditemukan:
   ✅ 'lolipp' mirip dengan 'lolipop' → 'Lolipop' (similarity 92.3%)

4. Hasil akhir koreksi: 'pak andi pesan Lolipop 10 dus'

=== MENGKOREKSI ENTRI 2: 'mses 5 bks' ===

1. Preprocessing angka

## Langkah 7: Ekstraksi Entitas dengan Regex

In [8]:
hasil_ekstraksi_list = []

for i, entry in enumerate(entries, 1):
    teks_koreksi = koreksi_teks(entry)
    hasil_ekstraksi = ekstrak_entitas(teks_koreksi, teks_asli=entry)
    
    hasil_ekstraksi_list.append(hasil_ekstraksi)
    
    print(f"\n--- ENTITAS ENTRI {i} ---")
    print(f"\nEntitas yang diekstrak:")
    
    entitas = hasil_ekstraksi['entitas']
    for key, value in entitas.items():
        if value not in (None, False, []):
            print(f"  {key:20s} : {value}")


--- ENTITAS ENTRI 1 ---

Entitas yang diekstrak:
  NAMA                 : Pak Andi
  AKSI                 : Tambah Penjualan
  BARANG               : Lolipop
  JUMLAH               : 10 dus
  SATUAN               : dus

--- ENTITAS ENTRI 2 ---

Entitas yang diekstrak:
  AKSI                 : Tambah Penjualan
  BARANG               : Meses
  JUMLAH               : 5 bungkus
  SATUAN               : bungkus

--- ENTITAS ENTRI 3 ---

Entitas yang diekstrak:
  AKSI                 : Tambah Penjualan
  BARANG               : Willo
  JUMLAH               : 3 dus
  SATUAN               : dus
  STATUS               : Hutang
  SEMUA                : True


## Langkah 8: Context Inheritance & Penyusunan Data Transaksi

In [9]:
print("\nLOGIKA:")
print("1. Sistem mencari konteks (nama, tanggal) dari entri pertama yang memilikinya.")
print("2. Lalu menerapkan ke entri lain yang KOSONG.")
print("3. Jika TANGGAL tidak ada, sistem otomatis gunakan tanggal HARI INI!")
print("4. Ini untuk kasus: satu pesanan, satu nama pembeli!")

# Simulasi langkah demi langkah
print("\n--- HASIL PENYUSUNAN DATA TRANSAKSI:")

# 1. Buat list hasil sementara
temp_results = []
for i, entry in enumerate(entries, 1):
    teks_koreksi = koreksi_teks(entry)
    hasil_ekstraksi = ekstrak_entitas(teks_koreksi, teks_asli=entry)
    temp_results.append({
        "entitas": hasil_ekstraksi['entitas'],
        "original_text": entry
    })

# 2. Terapkan _apply_multi_overrides
final_results = _apply_multi_overrides(temp_results)

# 3. Tambahkan tanggal hari ini secara otomatis
from datetime import datetime
today = datetime.now().strftime("%d-%m-%Y")
for trans in final_results:
    if not trans["entitas"].get("TANGGAL"):
        trans["entitas"]["TANGGAL"] = today

for i, trans in enumerate(final_results, 1):
    print(f"\n--- TRANSAKSI {i} ---")
    entitas = trans["entitas"]
    for key, value in entitas.items():
        if value not in (None, False, []):
            print(f"  {key:20s} : {value}")


LOGIKA:
1. Sistem mencari konteks (nama, tanggal) dari entri pertama yang memilikinya.
2. Lalu menerapkan ke entri lain yang KOSONG.
3. Jika TANGGAL tidak ada, sistem otomatis gunakan tanggal HARI INI!
4. Ini untuk kasus: satu pesanan, satu nama pembeli!

--- HASIL PENYUSUNAN DATA TRANSAKSI:

--- TRANSAKSI 1 ---
  TANGGAL              : 21-06-2026
  NAMA                 : Pak Andi
  AKSI                 : Tambah Penjualan
  BARANG               : Lolipop
  JUMLAH               : 10 dus
  SATUAN               : dus
  STATUS               : Hutang

--- TRANSAKSI 2 ---
  TANGGAL              : 21-06-2026
  NAMA                 : Pak Andi
  AKSI                 : Tambah Penjualan
  BARANG               : Meses
  JUMLAH               : 5 bungkus
  SATUAN               : bungkus
  STATUS               : Hutang

--- TRANSAKSI 3 ---
  TANGGAL              : 21-06-2026
  NAMA                 : Pak Andi
  AKSI                 : Tambah Penjualan
  BARANG               : Willo
  JUMLAH           

## Langkah 9: Perhitungan Total Harga & Penanganan Data Kosong

In [10]:
print("\n--- LOGIKA PERHITUNGAN TOTAL ---")
print("1. Cari harga satuan dari Master Data dengan fuzzy matching.")
print("2. Skip barang dengan harga 0!")
print("3. Jika satuan tidak cocok, gunakan satuan pertama yang tersedia untuk barang tersebut!")
print("4. Hitung: Total = Jumlah × Harga Satuan.")
print("5. Jika TANGGAL tidak ada, sistem otomatis gunakan tanggal HARI INI!")
print("6. Jika status 'Hutang' → Metode Pembayaran tidak wajib!")

# Dummy master data untuk demonstrasi (mirip master data asli!)
dummy_master_barang = [
    {"nama": "Lolipop", "satuan": "pack", "harga": 15000},  # Mapping: "permen" → "Lolipop"
    {"nama": "Lolipop", "satuan": "dus", "harga": 150000},
    {"nama": "Meses", "satuan": "bungkus", "harga": 14000},
    {"nama": "Meses", "satuan": "karton", "harga": 140000},
    {"nama": "Willo", "satuan": "dus", "harga": 504000}
]

# Simulasi perhitungan
temp_results = []
for i, entry in enumerate(entries, 1):
    teks_koreksi = koreksi_teks(entry)
    hasil_ekstraksi = ekstrak_entitas(teks_koreksi, teks_asli=entry)
    temp_results.append({
        "entitas": hasil_ekstraksi['entitas'],
        "original_text": entry
    })

final_results = _apply_multi_overrides(temp_results)

# Tambahkan tanggal hari ini
from datetime import datetime
today = datetime.now().strftime("%d-%m-%Y")
for trans in final_results:
    if not trans["entitas"].get("TANGGAL"):
        trans["entitas"]["TANGGAL"] = today

print("\n--- HASIL PERHITUNGAN TOTAL:")
for i, trans in enumerate(final_results, 1):
    ent = trans["entitas"]
    
    # Cari harga (mirip dengan cari_harga_default!)
    barang = ent.get("BARANG")
    satuan = ent.get("SATUAN")
    harga_satuan = None
    
    if barang:
        # Check for aliases first (permen → Lolipop)
        barang_lookup = barang.lower()
        if barang_lookup in KAMUS_ALIAS:
            barang_lookup = KAMUS_ALIAS[barang_lookup].lower()
        elif barang_lookup in ["permen"]:
            barang_lookup = "lolipop"
            
        # First try exact match nama + satuan
        for b in dummy_master_barang:
            if b["harga"] <= 0:
                continue  # Skip barang dengan harga <= 0!
            if b["nama"].lower() == barang_lookup and b["satuan"].lower() == (satuan.lower() if satuan else ""):
                harga_satuan = b["harga"]
                break
        
        # If not found, try just nama
        if not harga_satuan:
            for b in dummy_master_barang:
                if b["harga"] <= 0:
                    continue  # Skip barang dengan harga <= 0!
                if b["nama"].lower() == barang_lookup:
                    harga_satuan = b["harga"]
                    break
    
    # Hitung total
    total = None
    if harga_satuan and ent.get("JUMLAH"):
        m_jml = re.search(r"\d+", str(ent["JUMLAH"]))
        if m_jml:
            jml_num = int(m_jml.group())
            total = jml_num * harga_satuan
            ent["TOTAL"] = format_rupiah(total)
            if harga_satuan:
                ent["HARGA"] = format_rupiah(harga_satuan)
    
    # Tampilkan
    print(f"\nTRANSAKSI {i} - {barang}")
    print(f"  Tanggal       : {ent.get('TANGGAL')}")
    print(f"  Barang        : {barang}")
    print(f"  Jumlah        : {ent.get('JUMLAH')}")
    print(f"  Harga Satuan  : {format_rupiah(harga_satuan) if harga_satuan else '[KOSONG - tambahkan ke master data!]'}")
    print(f"  TOTAL         : {format_rupiah(total) if total else '[KOSONG]'}")
    print(f"  Status        : {ent.get('STATUS')}")
    


--- LOGIKA PERHITUNGAN TOTAL ---
1. Cari harga satuan dari Master Data dengan fuzzy matching.
2. Skip barang dengan harga 0!
3. Jika satuan tidak cocok, gunakan satuan pertama yang tersedia untuk barang tersebut!
4. Hitung: Total = Jumlah × Harga Satuan.
5. Jika TANGGAL tidak ada, sistem otomatis gunakan tanggal HARI INI!
6. Jika status 'Hutang' → Metode Pembayaran tidak wajib!

--- HASIL PERHITUNGAN TOTAL:

TRANSAKSI 1 - Lolipop
  Tanggal       : 21-06-2026
  Barang        : Lolipop
  Jumlah        : 10 dus
  Harga Satuan  : Rp 150.000
  TOTAL         : Rp 1.500.000
  Status        : Hutang

TRANSAKSI 2 - Meses
  Tanggal       : 21-06-2026
  Barang        : Meses
  Jumlah        : 5 bungkus
  Harga Satuan  : Rp 14.000
  TOTAL         : Rp 70.000
  Status        : Hutang

TRANSAKSI 3 - Willo
  Tanggal       : 21-06-2026
  Barang        : Willo
  Jumlah        : 3 dus
  Harga Satuan  : Rp 504.000
  TOTAL         : Rp 1.512.000
  Status        : Hutang


## Langkah 10: Full Processing & Penyusunan Data Transaksi

In [11]:
print("(Data siap dikirim ke database!)")
print("\nCATATAN: Demo ini tidak terhubung ke database, tapi logika sama persis!")

hasil_lengkap = proses_nlp(INPUT_TEKS)

# Tambahkan tanggal hari ini (seperti di sistem nyata!)
from datetime import datetime
today = datetime.now().strftime("%d-%m-%Y")
for trans in hasil_lengkap:
    if not trans["entitas"].get("TANGGAL"):
        trans["entitas"]["TANGGAL"] = today

print(f"\nInput: {INPUT_TEKS}")
print(f"\nJumlah transaksi: {len(hasil_lengkap)}")

for i, transaksi in enumerate(hasil_lengkap, 1):
    print(f"\n--- TRANSAKSI {i} ---")
    print(f"\n1. Intent: {transaksi['intent']}")
    print(f"\n2. Entitas:")
    for key, value in transaksi['entitas'].items():
        if value not in (None, False):
            print(f"   {key:20s} : {value}")
    # Check status
    if transaksi['entitas'].get('STATUS') == 'Hutang':
        print("\n   ⚠️ Metode Pembayaran tidak wajib karena status Hutang!")

(Data siap dikirim ke database!)

CATATAN: Demo ini tidak terhubung ke database, tapi logika sama persis!

Input: pak anDi Pesan l0lipp 10 dus, mses 5 bks, will0 3 dus belum lunwas semua

Jumlah transaksi: 3

--- TRANSAKSI 1 ---

1. Intent: Catat_Penjualan_Cicil

2. Entitas:
   TANGGAL              : 21-06-2026
   NAMA                 : Pak Andi
   AKSI                 : Tambah Penjualan
   BARANG               : Lolipop
   JUMLAH               : 10 dus
   SATUAN               : dus
   STATUS               : Hutang

   ⚠️ Metode Pembayaran tidak wajib karena status Hutang!

--- TRANSAKSI 2 ---

1. Intent: Catat_Penjualan_Cicil

2. Entitas:
   TANGGAL              : 21-06-2026
   NAMA                 : Pak Andi
   AKSI                 : Tambah Penjualan
   BARANG               : Meses
   JUMLAH               : 5 bungkus
   SATUAN               : bungkus
   STATUS               : Hutang

   ⚠️ Metode Pembayaran tidak wajib karena status Hutang!

--- TRANSAKSI 3 ---

1. Intent: Catat_Penj

## Ringkasan Logika

In [12]:
print("\nRingkasan Logika untuk Laporan:")
print("1. Input → Teks mentah dari user.")
print("2. Preprocessing → Cleaning, Lowercase, Tokenization, Normalization.")
print("3. Fuzzy Matching → Koreksi typo dengan Levenshtein Distance & Similarity Score.")
print("4. Entity Extraction → Ekstrak entitas dengan Regex.")
print("5. Context Inheritance → Wariskan nama/tanggal ke entri lain (multi-entry satu pesanan).")
print("6. Auto Tanggal → Jika TANGGAL kosong, sistem otomatis gunakan TANGGAL HARI INI!")
print("7. Perhitungan Total → Total = Jumlah × Harga Satuan.")
print("8. Master Data Lookup → Cari harga dari master data, skip barang dengan harga 0!")
print("9. Flexibel Satuan → Jika satuan tidak cocok, gunakan satuan pertama yang tersedia!")
print("10. Status Pembayaran → Jika 'Hutang' → Metode Pembayaran tidak wajib!")
print("11. Data Transaksi → Data terstruktur siap database!")


Ringkasan Logika untuk Laporan:
1. Input → Teks mentah dari user.
2. Preprocessing → Cleaning, Lowercase, Tokenization, Normalization.
3. Fuzzy Matching → Koreksi typo dengan Levenshtein Distance & Similarity Score.
4. Entity Extraction → Ekstrak entitas dengan Regex.
5. Context Inheritance → Wariskan nama/tanggal ke entri lain (multi-entry satu pesanan).
6. Auto Tanggal → Jika TANGGAL kosong, sistem otomatis gunakan TANGGAL HARI INI!
7. Perhitungan Total → Total = Jumlah × Harga Satuan.
8. Master Data Lookup → Cari harga dari master data, skip barang dengan harga 0!
9. Flexibel Satuan → Jika satuan tidak cocok, gunakan satuan pertama yang tersedia!
10. Status Pembayaran → Jika 'Hutang' → Metode Pembayaran tidak wajib!
11. Data Transaksi → Data terstruktur siap database!
